# Product Master

In [0]:
from pyspark.sql import functions as F

# Paths
bronze_path = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/product_master/"
target_table = "mini_project_team_a3t7.silver.product_master"

# Read Bronze
df_bronze = spark.read.format("delta").load(bronze_path)

# -------------------------------
# Data Cleaning & Standardization
# -------------------------------

df_clean = (
    df_bronze

    # Remove null primary key
    .filter(F.col("product_id").isNotNull())

    # Keep only active products
    .filter(F.col("status") == "active")

    # Standardization
    .withColumn("product_name", F.trim("product_name"))
    .withColumn("category", F.upper(F.trim("category")))
    .withColumn("subcategory", F.trim("subcategory"))
    .withColumn("brand", F.trim("brand"))

    # Data type enforcement
    .withColumn("effective_date", F.to_date("effective_date"))
    .withColumn("cost_price", F.col("cost_price").cast("double"))
    .withColumn("selling_price", F.col("selling_price").cast("double"))
    .withColumn("weight", F.col("weight").cast("double"))

    # Business validation
    .filter(F.col("selling_price") >= F.col("cost_price"))
    .filter(F.col("weight") > 0)

    # Metadata
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# -------------------------------
# Write to Silver Layer
# -------------------------------

(
    df_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

# Store master

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_path = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/Store_Master/"

df_bronze = spark.read.format("delta").load(bronze_path)

In [0]:
df_clean = (
    df_bronze
    
    # Remove null PK
    .filter(F.col("store_id").isNotNull())
    
    # Trim string columns
    .withColumn("store_name", F.trim("store_name"))
    .withColumn("region", F.trim("region"))
    .withColumn("city", F.trim("city"))
    .withColumn("store_type", F.trim("store_type"))
    
    # Standardize casing
    .withColumn("region", F.upper("region"))
    .withColumn("city", F.initcap("city"))
    .withColumn("store_type", F.upper("store_type"))
    
    # Remove duplicates (latest record per store_id)
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("store_id")
            .orderBy(F.col("_ingestion_timestamp").desc())
        )
    )
    .filter("rn = 1")
    .drop("rn")
    
    # Basic validation
    .filter(F.col("opening_date").isNotNull())
)

In [0]:
df_final = (
    df_clean
    .withColumn("store_age_years",
        F.floor(F.datediff(F.current_date(), F.col("opening_date")) / 365)
    )
)

In [0]:
# Saving as table
(
    df_final
    .write
    .format("delta")
    .mode("overwrite")   # or "append" if incremental
    .option("overwriteSchema", "true")
    .saveAsTable("mini_project_team_a3t7.silver.store_master")
)

# Supplier Master

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_path = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/supply_master/"

df_silver = (
    spark.read.format("delta").load(bronze_path)

    # Remove null PK
    .filter(F.col("supplier_id").isNotNull())

    # Basic cleaning
    .withColumn("supplier_name", F.trim("supplier_name"))
    .withColumn("category", F.upper(F.trim("category")))

    # Deduplicate (latest record per supplier_id)
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("supplier_id")
            .orderBy(F.col("_ingestion_timestamp").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Save as Silver table
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mini_project_team_a3t7.silver.supplier_master")
)